# 08 · Silver — `ai_classify` + `ai_extract`

**Silver job:** turn parse text into typed columns and land a table you can join.

- `ai_classify` labels the packet page: FNOL / invoice / adjuster / policy
- `ai_extract` pulls claim, pet, clinic, visit date, invoice, amount, diagnosis

Labels for `ai_classify` are a **JSON array string**, not `ARRAY()`. Both functions use version **2.0**. `extracted:response:<field>` is the cell to read.

Cap input with `substr(..., 1, 8000)` so a long parse cannot blow the context window.


In [ ]:
%run ./00_setup


In [ ]:
spark.sql("""
CREATE OR REPLACE TABLE silver_claim_documents
COMMENT 'Classified + extracted pet-claim PDF fields (AI Functions v2.0)'
AS
SELECT
  claim_id,
  file_name,
  path,
  parsed_text,
  ai_classify(
    substr(parsed_text, 1, 4000),
    '["fnol_claim_form", "clinic_invoice", "adjuster_report", "policy_declarations", "other"]',
    map('version', '2.0')
  ) AS classified,
  ai_extract(
    substr(parsed_text, 1, 8000),
    '{
      "claim_number": "Claim number such as CLM-PET-00021",
      "policy_number": "Policy number such as POL-88421",
      "pet_name": "Name of the pet",
      "species": "Dog or Cat",
      "breed": "Breed of the pet",
      "member_name": "Policyholder or clinic client name",
      "clinic_name": "Veterinary clinic or hospital name",
      "visit_date": "Visit date or date of loss as YYYY-MM-DD",
      "invoice_number": "Clinic invoice number if this is an invoice",
      "billed_amount": "Total billed amount as a number, no currency symbol",
      "diagnosis": "Diagnosis or presenting complaint",
      "recommended_decision": "Pay, Deny, or unknown"
    }',
    map('version', '2.0')
  ) AS extracted
FROM bronze_parsed_text
""")


In [ ]:
spark.sql("""
CREATE OR REPLACE TABLE silver_claim_documents_typed
COMMENT 'Silver typed columns from ai_classify + ai_extract :response'
AS
SELECT
  claim_id AS packet_claim_id,
  file_name,
  path,
  coalesce(
    classified:response:result::string,
    classified:result::string,
    trim(both '"' from CAST(classified AS STRING))
  ) AS doc_type,
  extracted:response:claim_number::string AS claim_number,
  extracted:response:policy_number::string AS policy_number,
  extracted:response:pet_name::string AS pet_name,
  extracted:response:species::string AS species,
  extracted:response:breed::string AS breed,
  extracted:response:member_name::string AS member_name,
  extracted:response:clinic_name::string AS clinic_name,
  try_to_date(extracted:response:visit_date::string) AS visit_date,
  extracted:response:invoice_number::string AS invoice_number,
  try_cast(extracted:response:billed_amount::string AS DOUBLE) AS billed_amount,
  extracted:response:diagnosis::string AS diagnosis,
  extracted:response:recommended_decision::string AS recommended_decision,
  classified,
  extracted
FROM silver_claim_documents
""")

display(spark.sql("""
SELECT
  packet_claim_id, doc_type, claim_number, pet_name, clinic_name,
  visit_date, invoice_number, billed_amount, diagnosis, recommended_decision
FROM silver_claim_documents_typed
ORDER BY packet_claim_id, doc_type
"""))


### Spot-check

You should see four `clinic_invoice` rows. Max appears twice at Pine Vet Clinic on `2026-08-03` with **different** invoice numbers. That is the planted duplicate.


In [ ]:
display(spark.sql("""
SELECT pet_name, clinic_name, visit_date, count(*) AS invoice_rows,
       collect_list(invoice_number) AS invoices,
       collect_list(claim_number) AS claims
FROM silver_claim_documents_typed
WHERE doc_type = 'clinic_invoice'
GROUP BY pet_name, clinic_name, visit_date
ORDER BY invoice_rows DESC, pet_name
"""))
